# Day 23: 分页 Cache 的逻辑与物理映射

logical_position // block_size 找 block table，余数找 offset。物理 block 可不连续，单个请求的逻辑顺序不变。

参考源码：`learning/mini_vllm/paged_kv_cache.py`。

每日概念、误区、面试问答：[知识讲义](../docs/KNOWLEDGE_DAY22_30.md)。

当前个人进度以[进度记录](../docs/LEARNING_PROGRESS.md)为准；运行参考答案不等于已独立掌握。

[Day 23完整讲义](../docs/DAY23_PAGED_ADDRESSING.md)：概念、计算契约、源码实验、调试优化、答案与资料。

先写自己的预测，再执行原有代码单元格；不要自动勾选个人验收。

## 1. 图解与数字例子

每个请求拿目录找自己的抽屉，而不是要求资料在大柜子里连成一段：

```text
block_size=4，table=[7,2,9]
请求位置0..3 → 物理块7
请求位置4..7 → 物理块2
请求位置8..  → 物理块9
```

逻辑位置6：`logical=6//4=1`，`offset=6%4=2`，
物理块为table[1]=2。不能把“逻辑块1”直接当“物理块1”。
位置编号从0开始，位置6其实是第7个位置。

## 先预测

执行前写下预期 shape、输出或调度顺序；在下面另建 Markdown 单元格保存自己的答案，不修改参考实现作为唯一证据。

In [ ]:
import json
import os
import sys
from pathlib import Path
root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/curriculum.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from learning.curriculum import run_day
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


In [ ]:
report = run_day(23, DEVICE)
print(json.dumps(report, indent=2, ensure_ascii=False))
destination = Path('logs') / f"day{report['day']:02d}-course.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n')
print('Report:', destination)


## 自测与失败实验

逻辑位置 6、block size 4 应映射到 table 的哪项/哪个 offset？

记录你改变的唯一变量、预期、实际结果和差异原因。

`requires_cuda` / `requires_cuda_compiler` / `external_service_not_configured` 都不是通过。

## 5. 自测与参考答案

**问：table=[7,2,9]读取顺序是2/7/9吗？**
答：不是，必须按逻辑目录7/2/9，还原请求历史顺序。

**问：目录能代替KV向量吗？**
答：不能，目录只是地址，数值在物理storage中。

**问：页越小越好吗？**
答：不一定，页尾浪费降低但表项和访问复杂度增加，需工作负载与硬件实测。

**问：这个分页器就是PagedAttention kernel吗？**
答：不是，管理器和读取/计算kernel是协作但不同的模块。

完整实现限制、调试步骤与资料见[本日讲义](../docs/DAY23_PAGED_ADDRESSING.md)。

以上是参考答案，不是个人学习结果。

## 验收记录

- [ ] 能解释原理
- [ ] 能自行复现
- [ ] 能解释一次失败
- [ ] 能说明当前实现不支持什么

个人笔记：

待填写。